In [1]:
# Importar librerías básicas
import pandas as pd
import numpy as np
import plotly.express as px
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# Algoritmos de clustering
from sklearn.cluster import KMeans

# Librerías de evaluación
from sklearn.metrics import silhouette_score, silhouette_samples

In [2]:
# Cargar los datos
path = 'https://raw.githubusercontent.com/jorgermzg15/data-files/refs/heads/main/Mall_Customers.csv'
df = pd.read_csv(path)

# Mostrar primeras filas
df.head()

,CustomerID,Genre,Age,Annual Income (k$),Spending Score (1-100)
0,1,Male,19,15,39
1,2,Male,21,15,81
2,3,Female,20,16,6
3,4,Female,23,16,77
4,5,Female,31,17,40


In [3]:
# Función para obtener mejor numero de clusters con silhouette score

def optimal_clusters_silhouette(X, max_k=10):
    best_score = -1
    best_k = 2
    for k in range(2, max_k + 1):
        kmeans = KMeans(n_clusters=k, random_state=42)
        labels = kmeans.fit_predict(X)
        score = silhouette_score(X, labels)
        print(f'K={k}, Silhouette Score={score:.4f}')
        if score > best_score:
            best_score = score
            best_k = k
    print(f'Optimal number of clusters: {best_k} with Silhouette Score: {best_score:.4f}')
    return best_k


scaler = StandardScaler()

# Convertir Genre a numerica
df['Genre Numeric'] = df['Genre'].map({'Male': 0, 'Female': 1})

X_scaled = scaler.fit_transform(df[['Annual Income (k$)', 'Spending Score (1-100)', 'Age']])
optimal_k = optimal_clusters_silhouette(X_scaled, max_k=10)

K=2, Silhouette Score=0.3355
K=3, Silhouette Score=0.3579
K=4, Silhouette Score=0.4040
K=5, Silhouette Score=0.4085
K=6, Silhouette Score=0.4311
K=7, Silhouette Score=0.4101
K=8, Silhouette Score=0.3674
K=9, Silhouette Score=0.3744
K=10, Silhouette Score=0.3619
Optimal number of clusters: 6 with Silhouette Score: 0.4311


In [4]:
kmeans = KMeans(n_clusters=6, random_state=42)
df['Cluster'] = kmeans.fit_predict(X_scaled)

# Visualización de clusters
fig = px.scatter_3d(
    df, 
    x='Age',
    y='Annual Income (k$)',
    z='Spending Score (1-100)',
    color='Cluster', 
    title='Clusters de Clientes'
)
fig.update_traces(marker=dict(size=5))
fig.update_layout(height=900)  # más alto
fig.show()

In [5]:
import plotly.graph_objects as go

# Invertir la escala para obtener los centroides en valores originales
centroids_original = scaler.inverse_transform(kmeans.cluster_centers_)
centroids_df = pd.DataFrame(centroids_original, columns=['Annual Income (k$)', 'Spending Score (1-100)', 'Age'])
centroids_df['Cluster'] = range(optimal_k)

# Usar la misma escala de color del gráfico original
colorscale = fig.data[0].marker.colorscale
cmin = df['Cluster'].min()
cmax = df['Cluster'].max()

fig.add_trace(go.Scatter3d(
    x=centroids_df['Age'],
    y=centroids_df['Annual Income (k$)'],
    z=centroids_df['Spending Score (1-100)'],
    mode='markers',
    marker=dict(
        symbol='x',
        size=5,
        color=centroids_df['Cluster'],
        colorscale=colorscale,
        cmin=cmin,
        cmax=cmax,
        line=dict(width=3),
    ),
    name='Centroides'
))

fig.show()
